# Phishing Domain Detection

## Problem statement
Phishing URLs attempt to impersonate trusted sites to obtain sensitive information. This notebook compares text-based classifiers for URL labels. Label `1` means phishing (`bad`); label `0` means `good`. The dataset has no usable timestamp/source metadata, so this workflow does not estimate temporal generalization.

## Imports and reproducibility

In [ ]:
import json
import hashlib
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.tokenize import RegexpTokenizer
from nltk.stem.snowball import SnowballStemmer
from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

RANDOM_SEED = 42
PHISHING_LABEL = 1

## Load and inspect the full dataset
The previous notebook capped the data at 5,000 rows but did not document a runtime or memory reason. The revised workflow uses all usable unique URL rows. No fuzzy/domain-level deduplication is applied.

In [ ]:
df = pd.read_csv("phishing_site_urls.csv")
print("Original rows:", len(df))
print("Original class counts (raw labels):\n", df["Label"].value_counts(dropna=False))
df.head()

In [ ]:
assert {"URL", "Label"}.issubset(df.columns)
assert df["URL"].notna().all()
assert set(df["Label"].dropna().unique()) <= {"good", "bad"}

df["Label"].value_counts().plot(kind="bar", title="Original label distribution")
plt.ylabel("Rows")
plt.show()

## Exact duplicates and conflicting labels
Exact duplicate URL strings are collapsed before splitting so repeated observations cannot overweight a URL or appear in multiple partitions. URLs on the same domain are retained as distinct examples. The one conflicting URL group (`tommyhumphreys.com/`) is excluded rather than resolved by majority vote because its label source cannot be adjudicated from this CSV.

In [ ]:
LABEL_MAP = {"good": 0, "bad": 1}
CONFLICTING_URLS = {"tommyhumphreys.com/"}

def prepare_dataset(dataframe, conflicting_urls=CONFLICTING_URLS):
    """Validate labels, report conflict groups, drop them, and deduplicate exact URLs."""
    label_counts = dataframe.groupby("URL")["Label"].nunique()
    conflicts = set(label_counts[label_counts > 1].index)
    if conflicts != set(conflicting_urls):
        raise ValueError(f"Unexpected conflicting URL groups: {conflicts}")
    conflict_rows = dataframe[dataframe["URL"].isin(conflicts)].copy()
    usable = dataframe[~dataframe["URL"].isin(conflicts)].drop_duplicates("URL", keep="first").copy()
    usable["Label"] = usable["Label"].map(LABEL_MAP).astype("int8")
    return usable, conflicts, conflict_rows

def split_dataset(dataframe, seed=42):
    """Return fixed 70/15/15 stratified splits over unique exact URLs."""
    train, remainder = train_test_split(dataframe, test_size=0.30, random_state=seed, stratify=dataframe["Label"])
    validation, test = train_test_split(remainder, test_size=0.50, random_state=seed, stratify=remainder["Label"])
    url_sets = [set(part["URL"]) for part in (train, validation, test)]
    assert url_sets[0].isdisjoint(url_sets[1])
    assert url_sets[0].isdisjoint(url_sets[2])
    assert url_sets[1].isdisjoint(url_sets[2])
    assert len(train) + len(validation) + len(test) == len(dataframe)
    return train, validation, test

def make_text_pipeline(tokenizer_function, classifier, vectorizer_class=CountVectorizer):
    """Build a pipeline whose vocabulary is learned only by fit on training text."""
    return Pipeline([
        ("vectorizer", vectorizer_class(
            tokenizer=tokenizer_function, token_pattern=None, lowercase=False
        )),
        ("classifier", classifier),
    ])

def make_cv(seed=42):
    """Three-fold shuffled stratification with a fixed seed."""
    return StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)


def build_candidate_config(seed=42):
    """Return active full-evaluation candidates and explicitly deferred models."""
    candidates = {
        "Logistic Regression": (
            LogisticRegression(max_iter=1000, random_state=seed),
            {"C": [0.1, 1.0, 10.0]},
        ),
        "LinearSVC": (
            LinearSVC(random_state=seed, max_iter=5000),
            {"C": [0.1, 1.0]},
        ),
        "Decision Tree": (
            DecisionTreeClassifier(random_state=seed),
            {"max_depth": [10, 25, None]},
        ),
        "Random Forest": (
            RandomForestClassifier(random_state=seed, n_jobs=-1),
            {"n_estimators": [100], "max_depth": [None, 25]},
        ),
        "XGBoost": (
            XGBClassifier(eval_metric="logloss", random_state=seed, n_jobs=-1),
            {"n_estimators": [100], "max_depth": [3, 6], "learning_rate": [0.1]},
        ),
    }
    deferred_candidates = {
        "Gradient Boosting": (
            GradientBoostingClassifier(random_state=seed),
            {"n_estimators": [100], "learning_rate": [0.1, 0.5], "max_depth": [2, 3]},
        ),
    }
    return candidates, deferred_candidates


In [ ]:
original_rows = len(df)
unique_url_count = df["URL"].nunique()
duplicate_extra_rows = original_rows - unique_url_count
conflicting_counts = df.groupby("URL")["Label"].nunique()
conflicting_urls = set(conflicting_counts[conflicting_counts > 1].index)
conflict_rows = df[df["URL"].isin(conflicting_urls)].copy()
assert conflicting_urls == CONFLICTING_URLS, f"Unexpected conflicts: {conflicting_urls}"

df_usable, conflicting_urls, conflict_rows = prepare_dataset(df)
print("Original row count:", original_rows)
print("Unique exact URL count:", unique_url_count)
print("Duplicate rows beyond first occurrence:", duplicate_extra_rows)
print("Repeated URL groups:", int((df.groupby("URL").size() > 1).sum()))
print("Conflicting URL groups excluded:", len(conflicting_urls))
print("Excluded URL label counts:")
print(conflict_rows.groupby(["URL", "Label"]).size())
print("Original class counts:", df["Label"].value_counts().sort_index().to_dict())
print("Usable class counts after exact deduplication and conflict exclusion (0=good, 1=phishing):",
      df_usable["Label"].value_counts().sort_index().to_dict())
print("Final usable rows:", len(df_usable))
DATASET_IDENTITY = hashlib.sha256(
    pd.util.hash_pandas_object(df_usable[["URL", "Label"]], index=False).values.tobytes()
).hexdigest()


## Reproducible train/validation/test split
Use a fixed, stratified 70/15/15 split of unique URLs. Exact URL deduplication before this split makes URL-level grouping inherent; explicit assertions below guard that invariant. Stratification preserves class proportions approximately. There is no temporal split because the dataset has no usable timestamp/source metadata.

In [ ]:
train_df, validation_df, test_df = split_dataset(df_usable, seed=RANDOM_SEED)

def report_split(name, partition):
    print(f"{name}: {len(partition)} rows ({len(partition) / len(df_usable):.1%})")
    counts = partition["Label"].value_counts().reindex([0, 1], fill_value=0)
    print("  counts (0=good, 1=phishing):", counts.to_dict())
    print("  proportions:", (counts / len(partition)).round(4).to_dict())

for split_name, split_df in [("Train", train_df), ("Validation", validation_df), ("Test", test_df)]:
    report_split(split_name, split_df)

def split_summary(partition):
    counts = partition["Label"].value_counts().reindex([0, 1], fill_value=0)
    return {"rows": len(partition), "class_counts": counts.to_dict()}


## Text preparation and training-only vocabulary\nTokenization and stemming are deterministic text transformations. During cross-validation, each Pipeline fits its CountVectorizer only on that fold's training rows. The selected Pipeline is refit on the full training partition and transforms validation/test without refitting its vocabulary.

In [ ]:
tokenizer = RegexpTokenizer(r"[A-Za-z]+")
stemmer = SnowballStemmer("english")

def prepare_url(text):
    return " ".join(stemmer.stem(token) for token in tokenizer.tokenize(text))

def tokenize_prepared_url(text):
    return prepare_url(text).split()

X_train_text = train_df["URL"]
X_validation_text = validation_df["URL"]
X_test_text = test_df["URL"]
y_train = train_df["Label"]
y_validation = validation_df["Label"]
y_test = test_df["Label"]
print("Raw URLs passed to each model pipeline; fold vectorizers are trained on training rows only.")


## Candidate models and validation-based selection
Selection prioritizes phishing recall to reduce false negatives, while also considering phishing precision and F1. False negatives are phishing URLs incorrectly classified as safe. Raising recall often increases false positives (safe URLs flagged), so the validation results must be interpreted as a trade-off, not ranked by accuracy alone. Test data is not referenced in this section. KNN remains omitted because repeated neighbor-distance calculations over hundreds of thousands of sparse rows are impractical. Gradient Boosting is retained in a deferred candidate configuration, but excluded from the initial full evaluation: its previous grid adds 12 CV fits plus a refit, and its full-data cost should be assessed after the primary candidate set. It can later be benchmarked and evaluated with the same train-only CV and validation policy.

## Fixed prediction policy
No custom decision threshold is selected and no threshold tuning is performed. Validation and held-out test predictions both use the selected estimator pipeline's same default `predict()` policy. Estimators that expose probabilities generally predict using their class/probability decision (commonly the 0.5 boundary for binary classifiers); margin-based estimators use their decision boundary. LinearSVC is a margin-based classifier: its signed decision boundary is not a probability threshold. No historical SVC metrics are directly comparable because LinearSVC uses a different solver and default loss/intercept treatment. The evaluation reports the estimator defaults without interpreting scores as calibrated probabilities.

### Candidate replacement note
LinearSVC replaces `SVC(kernel="linear")` as a distinct estimator. It uses a different optimization implementation and default loss/intercept treatment, so its results must be evaluated as a new candidate and not compared as if they were historical SVC results.

### Full-dataset execution safeguards
The next section uses all 507,194 usable rows and the fixed train/validation/test partitions; it must not silently substitute a sample. Full evaluation may require substantial time and memory, particularly for tree ensembles. The held-out test partition remains outside search and selection. Fold metrics, validation metrics, compatibility metadata and the final test result are stored as inspectable JSON under `evaluation_checkpoints/url_safety_seed_42`; fitted estimators and production model artifacts are never serialized. Each fold emits flushed progress, and completed compatible folds/validations are reused on restart. A dataset, split, candidate or preprocessing mismatch raises an error and requires a fresh checkpoint directory. The held-out test cell requires a finalized validation selection and reads its completion marker to avoid repeating an already completed test evaluation.

### Active search spaces

- **Logistic Regression:** `C` = 0.1, 1, 10; three log-spaced regularization strengths.
- **LinearSVC:** `C` = 0.1, 1; a compact low/default-strength comparison for the linear margin model.
- **Decision Tree:** `max_depth` = 10, 25, unlimited; compares shallow, deeper, and unrestricted trees without changing the existing range.
- **Random Forest:** 100 trees with `max_depth` = 25 or unlimited; isolates the depth constraint while keeping ensemble size fixed.
- **XGBoost:** 100 estimators, `learning_rate` = 0.1, `max_depth` = 3 or 6; a narrow depth comparison with the current learning rate and estimator count fixed.

All active grids are evaluated with three-fold shuffled `StratifiedKFold` on training rows. Grid searches run sequentially (`n_jobs=1` at the search level); Random Forest and XGBoost retain their estimator-level `n_jobs=-1`, so those workers are not nested inside concurrent searches. The deferred Gradient Boosting configuration is retained at 100 estimators, `learning_rate` = 0.1 or 0.5, and `max_depth` = 2 or 3. Its four combinations would add 12 CV fits plus one refit; benchmark it separately on training data before adding it to the full run.


In [ ]:
cv = make_cv(RANDOM_SEED)
candidates, deferred_candidates = build_candidate_config(RANDOM_SEED)

PREPROCESSING_CONFIG = {
    "tokenizer": "RegexpTokenizer(r'[A-Za-z]+') then SnowballStemmer('english')",
    "vectorizer": {"class": "sklearn.feature_extraction.text.CountVectorizer",
                   "token_pattern": None, "lowercase": False,
                   "input": "raw URL text passed through tokenizer"},
}
CHECKPOINT_DIRECTORY = Path("evaluation_checkpoints") / f"url_safety_seed_{RANDOM_SEED}"


def _json_safe(value):
    """Convert configuration values to deterministic JSON-compatible values."""
    if isinstance(value, np.generic):
        return _json_safe(value.item())
    if isinstance(value, dict):
        return {str(key): _json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if value is None or isinstance(value, (str, int, bool)):
        return value
    if isinstance(value, float):
        return value if np.isfinite(value) else str(value)
    if callable(value):
        return f"{value.__module__}.{getattr(value, '__qualname__', value.__name__)}"
    return f"{type(value).__module__}.{type(value).__qualname__}"


def _canonical_json(value):
    return json.dumps(_json_safe(value), sort_keys=True, separators=(",", ":"), allow_nan=False)


def _frame_fingerprint(texts, labels):
    frame = pd.DataFrame({"URL": np.asarray(texts, dtype=object), "Label": np.asarray(labels)})
    values = pd.util.hash_pandas_object(frame, index=False).values.tobytes()
    return hashlib.sha256(values).hexdigest()


def build_checkpoint_metadata(dataset_identity, training_text, training_labels,
                              validation_text, validation_labels, seed, candidates,
                              preprocessing_config):
    """Build metadata from dataset identity plus training/validation partitions only."""
    candidate_config = []
    for name, (estimator, grid) in candidates.items():
        candidate_config.append({
            "name": name,
            "estimator_class": f"{type(estimator).__module__}.{type(estimator).__qualname__}",
            "estimator_parameters": _json_safe(estimator.get_params(deep=False)),
            "grid": _json_safe(grid),
        })
    return {
        "schema_version": 1,
        "dataset_identity": str(dataset_identity),
        "split_metadata": {
            "seed": int(seed),
            "strategy": "stratified 70/15/15 split of deduplicated exact URLs",
            "training_rows": len(training_text),
            "training_identity": _frame_fingerprint(training_text, training_labels),
            "validation_rows": len(validation_text),
            "validation_identity": _frame_fingerprint(validation_text, validation_labels),
        },
        "candidate_configuration": candidate_config,
        "preprocessing_configuration": _json_safe(preprocessing_config),
    }


class CheckpointCompatibilityError(ValueError):
    """Raised when persisted metadata differs from this requested evaluation."""


class ResumableCheckpointStore:
    """JSON metrics/status checkpoints; fitted estimators are never serialized."""
    def __init__(self, directory, compatibility_metadata):
        self.directory = Path(directory)
        self.compatibility = _json_safe(compatibility_metadata)
        self.compatibility_digest = hashlib.sha256(
            _canonical_json(self.compatibility).encode("utf-8")
        ).hexdigest()
        manifest = self.directory / "manifest.json"
        if manifest.exists():
            try:
                existing = json.loads(manifest.read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError) as error:
                raise CheckpointCompatibilityError(
                    f"Checkpoint manifest is unreadable ({error}); select a fresh checkpoint directory."
                ) from error
            if existing != self.compatibility:
                raise CheckpointCompatibilityError(
                    "Checkpoint compatibility mismatch (dataset, split, candidate, or preprocessing "
                    "configuration changed). Existing files are untouched; select a fresh checkpoint "
                    "directory to start a new run."
                )
        else:
            if self.directory.exists() and any(self.directory.iterdir()):
                raise CheckpointCompatibilityError(
                    "Checkpoint directory has files but no valid manifest; select a fresh directory."
                )
            self.directory.mkdir(parents=True, exist_ok=True)
            self._atomic_write(manifest, self.compatibility)
        self.fold_directory = self.directory / "folds"
        self.validation_directory = self.directory / "validation"
        self.fold_directory.mkdir(parents=True, exist_ok=True)
        self.validation_directory.mkdir(parents=True, exist_ok=True)

    def _atomic_write(self, path, value):
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".tmp")
        with temporary.open("w", encoding="utf-8", newline="\n") as handle:
            json.dump(_json_safe(value), handle, sort_keys=True, indent=2, allow_nan=False)
            handle.write("\n")
            handle.flush()
        temporary.replace(path)

    def fold_path(self, candidate, parameters, fold_number, seed):
        identity = {"candidate": candidate, "parameters": _json_safe(parameters),
                    "fold": int(fold_number), "seed": int(seed)}
        key = hashlib.sha256(_canonical_json(identity).encode("utf-8")).hexdigest()
        return self.fold_directory / f"{key}.json"

    def save_fold(self, candidate, parameters, fold_number, seed, status,
                  metrics=None, duration_seconds=0.0, error=None):
        record = {
            "compatibility_digest": self.compatibility_digest,
            "candidate": candidate, "parameters": _json_safe(parameters),
            "fold": int(fold_number), "seed": int(seed), "status": status,
            "metrics": _json_safe(metrics), "duration_seconds": float(duration_seconds),
        }
        if error is not None:
            record["error"] = _json_safe(error)
        self._atomic_write(self.fold_path(candidate, parameters, fold_number, seed), record)
        return record

    def load_completed_fold(self, candidate, parameters, fold_number, seed):
        path = self.fold_path(candidate, parameters, fold_number, seed)
        if not path.exists():
            return None
        try:
            record = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError) as error:
            print(f"Invalid fold checkpoint {path.name}; rerunning this fold: {error}", flush=True)
            return None
        valid_identity = (
            record.get("compatibility_digest") == self.compatibility_digest
            and record.get("candidate") == candidate
            and record.get("parameters") == _json_safe(parameters)
            and record.get("fold") == int(fold_number)
            and record.get("seed") == int(seed)
        )
        if valid_identity and record.get("status") == "complete" and isinstance(record.get("metrics"), dict):
            return record
        return None

    def count_failed_folds(self):
        count = 0
        for path in self.fold_directory.glob("*.json"):
            try:
                record = json.loads(path.read_text(encoding="utf-8"))
                count += int(record.get("compatibility_digest") == self.compatibility_digest
                             and record.get("status") == "failed")
            except (OSError, json.JSONDecodeError):
                continue
        return count

    def count_completed_fold_records(self):
        count = 0
        for path in self.fold_directory.glob("*.json"):
            try:
                record = json.loads(path.read_text(encoding="utf-8"))
                count += int(record.get("compatibility_digest") == self.compatibility_digest
                             and record.get("status") == "complete")
            except (OSError, json.JSONDecodeError):
                continue
        return count

    def count_completed_validation_records(self):
        count = 0
        for path in self.validation_directory.glob("*.json"):
            try:
                record = json.loads(path.read_text(encoding="utf-8"))
                count += int(record.get("compatibility_digest") == self.compatibility_digest
                             and record.get("status") == "complete")
            except (OSError, json.JSONDecodeError):
                continue
        return count

    def validation_path(self, candidate, parameters):
        identity = {"candidate": candidate, "parameters": _json_safe(parameters)}
        key = hashlib.sha256(_canonical_json(identity).encode("utf-8")).hexdigest()
        return self.validation_directory / f"{key}.json"

    def load_validation(self, candidate, parameters):
        path = self.validation_path(candidate, parameters)
        if not path.exists():
            return None
        try:
            record = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError) as error:
            print(f"Invalid validation checkpoint {path.name}; rerunning candidate validation: {error}", flush=True)
            return None
        valid_identity = (
            record.get("compatibility_digest") == self.compatibility_digest
            and record.get("candidate") == candidate
            and record.get("parameters") == _json_safe(parameters)
            and record.get("status") == "complete"
        )
        return record if valid_identity else None

    def save_validation(self, candidate, parameters, result):
        record = {"compatibility_digest": self.compatibility_digest,
                  "candidate": candidate, "parameters": _json_safe(parameters),
                  "status": "complete", "result": _json_safe(result)}
        self._atomic_write(self.validation_path(candidate, parameters), record)
        return record

    def save_validation_failure(self, candidate, parameters, error):
        record = {"compatibility_digest": self.compatibility_digest,
                  "candidate": candidate, "parameters": _json_safe(parameters),
                  "status": "failed", "error": _json_safe(error)}
        self._atomic_write(self.validation_path(candidate, parameters), record)

    def finalize_selection(self, selected_name, selected_parameters, priority_metrics):
        selection = {
            "compatibility_digest": self.compatibility_digest,
            "status": "finalized", "selected_candidate": selected_name,
            "selected_parameters": _json_safe(selected_parameters),
            "priority_metrics": _json_safe(priority_metrics),
            "selection_priority": ["phishing_recall", "phishing_f1", "phishing_precision"],
        }
        path = self.directory / "selection.json"
        if path.exists():
            existing = json.loads(path.read_text(encoding="utf-8"))
            if existing != selection:
                raise CheckpointCompatibilityError(
                    "Finalized selection differs from stored selection; use a fresh checkpoint directory."
                )
        else:
            self._atomic_write(path, selection)
        return selection

    def load_finalized_selection(self):
        path = self.directory / "selection.json"
        if not path.exists():
            return None
        selection = json.loads(path.read_text(encoding="utf-8"))
        if (selection.get("compatibility_digest") != self.compatibility_digest
                or selection.get("status") != "finalized"):
            raise CheckpointCompatibilityError("Selection marker is invalid for this configuration.")
        return selection

    def load_completed_test(self):
        path = self.directory / "held_out_test.json"
        if not path.exists():
            return None
        try:
            record = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            return None
        valid = (record.get("compatibility_digest") == self.compatibility_digest
                 and record.get("status") == "complete")
        return record if valid else None

    def save_test_status(self, status, selected_name, selected_parameters,
                         metrics=None, error=None):
        record = {"compatibility_digest": self.compatibility_digest,
                  "status": status, "selected_candidate": selected_name,
                  "selected_parameters": _json_safe(selected_parameters),
                  "metrics": _json_safe(metrics)}
        if error is not None:
            record["error"] = _json_safe(error)
        self._atomic_write(self.directory / "held_out_test.json", record)
        return record


def metric_summary(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "phishing_precision": float(precision_score(y_true, y_pred, pos_label=PHISHING_LABEL, zero_division=0)),
        "phishing_recall": float(recall_score(y_true, y_pred, pos_label=PHISHING_LABEL, zero_division=0)),
        "phishing_f1": float(f1_score(y_true, y_pred, pos_label=PHISHING_LABEL, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
    }


def _take_rows(values, indices):
    return values.iloc[indices] if hasattr(values, "iloc") else np.asarray(values)[indices]


def _emit_fit_progress(progress_callback, event):
    if progress_callback is not None:
        progress_callback(event)
        return
    eta = event.get("estimated_remaining_seconds")
    eta_text = "n/a" if eta is None else f"~{eta:.1f}s (estimate)"
    print(
        "Progress: {overall_completed}/{planned_total_fits} completed; candidate={candidate}; "
        "params={parameters}; fold={fold}; fit={fit_duration_seconds:.2f}s; "
        "cumulative={cumulative_elapsed_seconds:.1f}s; failed={failed_fit_count}; "
        "remaining={remaining_fits}; ETA={eta}".format(eta=eta_text, **event),
        flush=True,
    )


def run_resumable_cv(candidates, training_text, training_labels, cv,
                     tokenizer_function, checkpoint_store, seed=42,
                     vectorizer_class=CountVectorizer, planned_total_fits=None,
                     progress_callback=None, cumulative_start_time=None):
    """Fit one train-only CV fold at a time, saving its metrics before proceeding."""
    parameter_sets = {name: list(ParameterGrid(grid)) for name, (_, grid) in candidates.items()}
    planned_cv_fits = sum(map(len, parameter_sets.values())) * cv.get_n_splits()
    planned_total_fits = planned_total_fits or planned_cv_fits
    prior_validation = checkpoint_store.count_completed_validation_records()
    completed_cv = 0
    failed_count = checkpoint_store.count_failed_folds()
    durations = []
    run_start = cumulative_start_time or time.perf_counter()
    splits = list(cv.split(np.zeros(len(training_labels)), training_labels))
    cv_results = {}
    for name, (base_estimator, _) in candidates.items():
        candidate_results = []
        for parameters in parameter_sets[name]:
            fold_metrics = []
            for fold_number, (fold_train, fold_valid) in enumerate(splits, start=1):
                cached = checkpoint_store.load_completed_fold(name, parameters, fold_number, seed)
                if cached is not None:
                    metrics = cached["metrics"]
                    fit_duration = float(cached.get("duration_seconds", 0.0))
                    status = "checkpoint_reused"
                    durations.append(fit_duration)
                else:
                    pipeline = make_text_pipeline(
                        tokenizer_function, clone(base_estimator).set_params(**parameters),
                        vectorizer_class=vectorizer_class,
                    )
                    started = time.perf_counter()
                    try:
                        pipeline.fit(_take_rows(training_text, fold_train),
                                     _take_rows(training_labels, fold_train))
                        predictions = pipeline.predict(_take_rows(training_text, fold_valid))
                        metrics = metric_summary(_take_rows(training_labels, fold_valid), predictions)
                        fit_duration = time.perf_counter() - started
                        checkpoint_store.save_fold(name, parameters, fold_number, seed,
                                                   "complete", metrics, fit_duration)
                        status = "completed"
                        durations.append(fit_duration)
                    except Exception as error:
                        fit_duration = time.perf_counter() - started
                        failed_count += 1
                        checkpoint_store.save_fold(
                            name, parameters, fold_number, seed, "failed",
                            duration_seconds=fit_duration,
                            error={"type": type(error).__name__, "message": str(error)},
                        )
                        _emit_fit_progress(progress_callback, {
                            "overall_completed": completed_cv + prior_validation,
                            "planned_total_fits": planned_total_fits,
                            "candidate": name, "parameters": _json_safe(parameters),
                            "fold": fold_number, "fit_duration_seconds": fit_duration,
                            "cumulative_elapsed_seconds": time.perf_counter() - run_start,
                            "failed_fit_count": failed_count,
                            "remaining_fits": max(0, planned_total_fits-completed_cv-prior_validation),
                            "estimated_remaining_seconds": None, "status": "failed",
                        })
                        raise
                fold_metrics.append(metrics)
                completed_cv += 1
                overall = completed_cv + prior_validation
                remaining = max(0, planned_total_fits - overall)
                eta = float(np.mean(durations)) * remaining if durations else None
                _emit_fit_progress(progress_callback, {
                    "overall_completed": overall, "planned_total_fits": planned_total_fits,
                    "candidate": name, "parameters": _json_safe(parameters),
                    "fold": fold_number, "fit_duration_seconds": fit_duration,
                    "cumulative_elapsed_seconds": time.perf_counter()-run_start,
                    "failed_fit_count": failed_count, "remaining_fits": remaining,
                    "estimated_remaining_seconds": eta, "status": status,
                })
            candidate_results.append({
                "parameters": _json_safe(parameters),
                "cv_phishing_recall": float(np.mean([m["phishing_recall"] for m in fold_metrics])),
                "fold_metrics": fold_metrics,
            })
        best = max(candidate_results, key=lambda result: result["cv_phishing_recall"])
        cv_results[name] = candidate_results
        print(f"CV candidate summary: {name}; best parameters={best['parameters']}; "
              f"mean three-fold phishing recall={best['cv_phishing_recall']:.6f}", flush=True)
    return cv_results


def evaluate_candidates(candidates, cv_results, training_text, training_labels,
                        validation_text, validation_labels, tokenizer_function,
                        checkpoint_store, planned_total_fits,
                        progress_callback=None, vectorizer_class=CountVectorizer, cumulative_start_time=None):
    """Use train-only CV then validation; persist each candidate before selection."""
    rows, fitted = [], {}
    completed_validation = checkpoint_store.count_completed_validation_records()
    failed_count = checkpoint_store.count_failed_folds()
    run_start = cumulative_start_time or time.perf_counter()
    for name, (base_estimator, _) in candidates.items():
        best_cv = max(cv_results[name], key=lambda result: result["cv_phishing_recall"])
        parameters = best_cv["parameters"]
        cached = checkpoint_store.load_validation(name, parameters)
        if cached is not None:
            result, model, status, duration = cached["result"], None, "validation_checkpoint_reused", 0.0
        else:
            model = make_text_pipeline(
                tokenizer_function, clone(base_estimator).set_params(**parameters),
                vectorizer_class=vectorizer_class,
            )
            started = time.perf_counter()
            try:
                model.fit(training_text, training_labels)
                result = {
                    "model": name, "best_parameters": _json_safe(parameters),
                    "cv_phishing_recall": best_cv["cv_phishing_recall"],
                    "train": metric_summary(training_labels, model.predict(training_text)),
                    "validation": metric_summary(validation_labels, model.predict(validation_text)),
                }
                duration = time.perf_counter() - started
                checkpoint_store.save_validation(name, parameters, result)
                fitted[name] = model
                completed_validation += 1
                status = "validation_completed"
            except Exception as error:
                checkpoint_store.save_validation_failure(name, parameters,
                    {"type": type(error).__name__, "message": str(error)})
                raise
        rows.append(result)
        vm = result["validation"]
        print(f"Validation candidate summary: {name}; params={parameters}; "
              f"recall={vm['phishing_recall']:.6f}; F1={vm['phishing_f1']:.6f}; "
              f"precision={vm['phishing_precision']:.6f}; status={status}", flush=True)
        overall = checkpoint_store.count_completed_fold_records() + completed_validation
        remaining = max(0, planned_total_fits-overall)
        _emit_fit_progress(progress_callback, {
            "overall_completed": overall, "planned_total_fits": planned_total_fits,
            "candidate": name, "parameters": _json_safe(parameters), "fold": "refit/validation",
            "fit_duration_seconds": duration, "cumulative_elapsed_seconds": time.perf_counter()-run_start,
            "failed_fit_count": failed_count, "remaining_fits": remaining,
            "estimated_remaining_seconds": None, "status": status,
        })
    selected = max(rows, key=lambda row: (row["validation"]["phishing_recall"],
                                          row["validation"]["phishing_f1"],
                                          row["validation"]["phishing_precision"]))
    checkpoint_store.finalize_selection(selected["model"], selected["best_parameters"], selected["validation"])
    selected_model = fitted.get(selected["model"])
    if selected_model is None:
        selected_model = make_text_pipeline(
            tokenizer_function,
            clone(candidates[selected["model"]][0]).set_params(**selected["best_parameters"]),
            vectorizer_class=vectorizer_class,
        )
        # JSON checkpoints deliberately omit estimators; a resumed run rebuilds the winner on train only.
        selected_model.fit(training_text, training_labels)
    table = pd.DataFrame([
        {"model": row["model"], "best_parameters": row["best_parameters"],
         "cv_phishing_recall": row["cv_phishing_recall"],
         **{f"train_{k}": v for k, v in row["train"].items() if k != "confusion_matrix"},
         **{f"validation_{k}": v for k, v in row["validation"].items() if k != "confusion_matrix"}}
        for row in rows
    ]).sort_values(["validation_phishing_recall", "validation_phishing_f1",
                    "validation_phishing_precision"], ascending=False)
    print("Selection finalized by validation phishing recall, then F1, then precision: "
          f"{selected['model']} {selected['best_parameters']}", flush=True)
    return rows, selected_model, selected["model"], selected["best_parameters"], table


def evaluate_held_out_once(selected_estimator, test_text, test_labels,
                           checkpoint_store, selected_name, selected_parameters):
    """Require finalized selection and persist a one-time held-out test result."""
    selection = checkpoint_store.load_finalized_selection()
    if selection is None:
        raise RuntimeError("Held-out test evaluation is blocked until model selection is finalized.")
    if (selection["selected_candidate"] != selected_name
            or selection["selected_parameters"] != _json_safe(selected_parameters)):
        raise CheckpointCompatibilityError("Test candidate differs from the finalized selection.")
    completed = checkpoint_store.load_completed_test()
    if completed is not None:
        return None, completed["metrics"]
    if selected_estimator is None:
        raise RuntimeError("A selected fitted estimator is required for the first held-out evaluation.")
    checkpoint_store.save_test_status("running", selected_name, selected_parameters)
    try:
        predictions = selected_estimator.predict(test_text)
        metrics = metric_summary(test_labels, predictions)
    except Exception as error:
        checkpoint_store.save_test_status("failed", selected_name, selected_parameters,
            error={"type": type(error).__name__, "message": str(error)})
        raise
    checkpoint_store.save_test_status("complete", selected_name, selected_parameters, metrics)
    return predictions, metrics


# Checkpoints contain JSON configuration, metrics and status only, never fitted models.
checkpoint_metadata = build_checkpoint_metadata(
    DATASET_IDENTITY, X_train_text, y_train, X_validation_text, y_validation,
    RANDOM_SEED, candidates, PREPROCESSING_CONFIG,
)
checkpoint_store = ResumableCheckpointStore(CHECKPOINT_DIRECTORY, checkpoint_metadata)
cv_fit_plan = {name: len(list(ParameterGrid(grid))) * cv.get_n_splits()
               for name, (_, grid) in candidates.items()}
planned_cv_fits = sum(cv_fit_plan.values())
planned_validation_refits = len(candidates)
planned_total_fits = planned_cv_fits + planned_validation_refits
print("Dataset rows:", len(df_usable))
print("Train / validation / untouched test rows:", len(train_df), len(validation_df), len(test_df))
print("CV fits by candidate:", cv_fit_plan)
print("Planned CV folds:", planned_cv_fits)
print("Planned candidate refits/validation evaluations:", planned_validation_refits)
print("Total planned search and validation fits:", planned_total_fits)
print("Checkpoint directory:", CHECKPOINT_DIRECTORY)
print("Full evaluation may require substantial time and memory; no downsampling will occur.")
print("Only JSON metrics/status are saved; production model artifacts are not written.", flush=True)

evaluation_started_at = time.perf_counter()
cv_results = run_resumable_cv(
    candidates, X_train_text, y_train, cv, tokenize_prepared_url,
    checkpoint_store, seed=RANDOM_SEED, planned_total_fits=planned_total_fits,
    cumulative_start_time=evaluation_started_at,
)
selection_rows, selected_pipeline, selected_name, selected_parameters, selection_table = evaluate_candidates(
    candidates, cv_results, X_train_text, y_train, X_validation_text, y_validation,
    tokenize_prepared_url, checkpoint_store, planned_total_fits,
    cumulative_start_time=evaluation_started_at,
)
selection_table

In [ ]:
for row in selection_rows:
    print("\n", row["model"], "parameters:", row["best_parameters"])
    print("Train metrics:", row["train"])
    print("Validation metrics:", row["validation"])
    print("Validation confusion matrix [true rows good/phishing, predicted columns good/phishing]:\n",
          np.array(row["validation"]["confusion_matrix"]))
print("\nSelected using validation phishing recall, then F1, then precision:", selected_name)


## One-time held-out test evaluation
The selected model and parameters are finalized using training cross-validation and validation metrics only. The test helper requires the persisted final-selection marker. It writes held-out metrics to a separate JSON completion record only after a successful evaluation; resuming a completed run reads those metrics and does not call `predict` again. Do not use test results to revise the model, parameters, features, or thresholds.

In [ ]:
held_out_predictions, held_out_metrics = evaluate_held_out_once(
    selected_pipeline, X_test_text, y_test, checkpoint_store,
    selected_name, selected_parameters,
)
print("HELD-OUT TEST RESULTS - final evaluation only")
print(held_out_metrics)
print("Confusion matrix [true rows good/phishing, predicted columns good/phishing]:\n",
      np.array(held_out_metrics["confusion_matrix"]))
print("False negatives (phishing incorrectly classified as safe):",
      held_out_metrics["confusion_matrix"][1][0])
ConfusionMatrixDisplay(
    confusion_matrix=np.array(held_out_metrics["confusion_matrix"]),
    display_labels=["good", "phishing"]
).plot(cmap="Blues")
plt.title("Held-out test confusion matrix")
plt.show()

## Interpretation
Phishing recall is the fraction of phishing URLs detected; low recall means more false negatives that appear safe. Precision is the fraction of URLs flagged as phishing that are actually phishing; increasing recall may reduce precision and flag more safe URLs. Choose operational thresholds only with validation data and a stated cost trade-off. The held-out test results above are final and must not guide further model changes.